# btp-resident-validation
Fresh M0 repeatability and resident A-B-A-B gate. Part of the approved development batch; no held-out scenes or teacher targets.

Implementation workflow; no completed GPU experiment is implied. Historical notebooks are preserved.

## Source and runtime
Attach the private research source bundle. Internet is needed to provision the isolated runtime; running this cell downloads dependencies. CPU smoke precedes GPU use. No long experiment runs automatically.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working") / ("btp-research-" + str(time.time_ns()))
OUTPUT.mkdir()
# Upload artifacts/btp-kaggle.zip as btp-research-source.bundle in a private dataset.
bundles = list(Path("/kaggle/input").rglob("btp-research-source.bundle"))
if not ROOT.exists():
    assert len(bundles) == 1, (
        "Attach one research source bundle; see docs/RESEARCH_WORKFLOWS.md"
    )
    with zipfile.ZipFile(bundles[0]) as archive:
        for info in archive.infolist():
            destination = (ROOT.parent / info.filename).resolve()
            assert destination.is_relative_to(ROOT), "Unsafe source member"
        archive.extractall(ROOT.parent)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
if len(bundles) == 1:
    with zipfile.ZipFile(bundles[0]) as archive:
        assert snapshot == json.loads(archive.read("BTP/source-manifest.json")), (
            "Stale working source: start a fresh session or preserve and move the old BTP directory"
        )
for name, expected in snapshot["files"].items():
    path = (ROOT / name).resolve()
    assert path.is_relative_to(ROOT) and path.is_file(), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
print("Verified source files:", len(snapshot["files"]))

In [ ]:
uv = shutil.which("uv")
if uv is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [uv] if uv else [sys.executable, "-m", "uv"]
PYTHON = Path("/tmp/btp-research-env/bin/python")
if not PYTHON.exists():
    subprocess.run(
        UV + ["venv", "--python", "3.10", str(PYTHON.parent.parent)], check=True
    )
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0")


def run(arguments):
    subprocess.run(RUN + arguments, cwd=ROOT, env=ENVIRONMENT, check=True)


with (OUTPUT / "runtime-packages.txt").open("w") as handle:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], stdout=handle, check=True
    )
(OUTPUT / "source-manifest.json").write_text(json.dumps(snapshot, indent=2))

In [ ]:
archives = list(Path("/kaggle/input").rglob("nh-haze-512.bundle"))
assert len(archives) == 1
DATA = OUTPUT / "evaluation"
DATA.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        assert (DATA / name).resolve().is_relative_to(DATA)
    archive.extractall(DATA)
MANIFEST = DATA / "nh_haze_manifest.json"
assert (
    MANIFEST.read_bytes()
    == (ROOT / "configs/evaluation/nh_haze_manifest.json").read_bytes()
)
WEIGHTS = Path("/tmp") / (OUTPUT.name + "-weights-links")
WEIGHTS.mkdir()
catalog = json.loads((ROOT / "configs/checkpoints.json").read_text())
for name in ("v2-1_512-ema-pruned.ckpt", "stage2.pt"):
    matches = list(Path("/kaggle/input").rglob(name))
    assert len(matches) == 1, name
    digest = hashlib.sha256()
    with matches[0].open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    assert digest.hexdigest() == catalog[name]["sha256"], name
    (WEIGHTS / name).symlink_to(matches[0])

In [ ]:
run(
    [
        "-m",
        "btp.resident_validation",
        "--manifest",
        str(MANIFEST),
        "--weights",
        str(WEIGHTS),
        "--output",
        str(OUTPUT / "resident-validation"),
        "--max-seconds",
        "2400",
    ]
)
decision = json.loads((OUTPUT / "resident-validation/decision.json").read_text())
print("Status:", decision["status"], "Eligible:", decision["eligible"])
# Remove only local links before archive packaging; pretrained bytes remain on input mount.
for link in WEIGHTS.iterdir():
    assert link.is_symlink()
    link.unlink()
WEIGHTS.rmdir()

In [ ]:
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download evidence:", archive)